In [1]:
import os
os.chdir('../src')

In [6]:
import pandas as pd
import numpy as np

##### Synthesize data

In [ ]:


# Create synthetic multi-dimensional retail data
data = {
    'Date': pd.date_range(start='2024-01-01', periods=500, freq='D').repeat(2).tolist()[:500],
    'Location': np.random.choice(['New York', 'London', 'Tokyo', 'Berlin'], 500),
    'Store_ID': np.random.choice(['ST-001', 'ST-002', 'ST-003', 'ST-004'], 500),
    'Product_Category': np.random.choice(['Electronics', 'Apparel', 'Home & Kitchen', 'Groceries'], 500),
    'Sales_USD': np.random.uniform(100, 5000, 500).round(2),
    'Quantity': np.random.randint(1, 50, 500),
    'Discount_Applied': np.random.choice([True, False], 500, p=[0.3, 0.7])
}

df = pd.DataFrame(data)
df.to_csv('data/data.csv', index=False)
print("Dataset 'data.csv' created successfully!")

Dataset 'data.csv' created successfully!


#### Create a sample agent !

In [21]:
from dotenv import load_dotenv
load_dotenv()
grok_api_key = os.getenv("GROK_API_KEY")

#### Profiler node

In [15]:
from langchain_openai import ChatOpenAI
import io

llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=grok_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)



def profiler_node(state: dict):
    # 1. Extraction of inputs from State
    file_path = state.get("file_path")
    user_context = state.get("user_context", "No context provided")
    task = state.get("task_type", "General EDA")
    target = state.get("target_col", "Not specified")
    
    # Load data
    df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
    
    # 2. Programmatic Metadata (The "Truth")
    # We find potential grains by checking columns where unique count == row count
    total_rows = len(df)
    potential_pks = [col for col in df.columns if df[col].nunique() == total_rows]
    
    metadata = {
        "columns": df.columns.tolist(),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "uniques": df.nunique().to_dict(),
        "nulls": df.isnull().sum().to_dict(),
        "sample": df.head(3).to_markdown()
    }

    # 3. The Profiler Prompt
    prompt = f"""
    Act as an expert Data Profiler. 
    USER CONTEXT: {user_context}
    TASK: {task} | TARGET: {target}

    DATA FACTS:
    - Shape: {df.shape}
    - Potential Primary Keys: {potential_pks}
    - Metadata: {metadata['dtypes']}
    - Nulls: {metadata['nulls']}
    
    SAMPLE DATA:
    {metadata['sample']}

    YOUR MISSION:
    1. **Column Logic**: Explain each column's significance. Label them as (ID, Feature, or Target).
    2. **Granularity**: Determine the 'Grain' of the data. Is it one row per Customer? Per Transaction? Per Timestamp?
    3. **Suitability**: Briefly state if this data grain matches the user's goal of {task}.

    Output in clean Markdown.
    """

    response = llm.invoke(prompt)

    return {
        "data_profile": response.content,
        "messages": ["Data profiling and granularity detection completed."]
    }

In [16]:
# MOCK INPUTS
mock_state = {
    "file_path": "data/data.csv",
    "user_context": "I want to analyze retail performance across different stores.",
    "task_type": "Regression",
    "target_col": "Sales_USD",
    "messages": []
}

# RUN NODE
print("🛠️ Testing Profiler Node...")
result = profiler_node(mock_state)

print("\n" + "="*30)
print("PROFILER OUTPUT")
print("="*30)
print(result["data_profile"])

🛠️ Testing Profiler Node...

PROFILER OUTPUT
### Data Profiling Report
#### Column Logic
The given dataset contains 7 columns, each with its own significance. Here's a breakdown of each column:

* **Date**: Feature - This column represents the date of the sales transaction.
* **Location**: Feature - This column indicates the location of the store where the sales transaction took place.
* **Store_ID**: Feature - This column uniquely identifies the store where the sales transaction occurred.
* **Product_Category**: Feature - This column categorizes the type of product sold.
* **Sales_USD**: Target - This column represents the total sales amount in USD, which is the target variable for the regression task.
* **Quantity**: Feature - This column indicates the number of products sold.
* **Discount_Applied**: Feature - This column is a boolean indicator of whether a discount was applied to the sales transaction.

#### Granularity
The 'Grain' of the data appears to be one row per **sales trans

### Auditor node

In [22]:
def auditor_node(state: dict):
    # 1. Setup
    file_path = state.get("file_path")
    task = state.get("task_type", "General EDA")
    target = state.get("target_col")
    df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
    
    # 2. MATHEMATICAL CALCULATIONS (The "Hard" Evidence)
    
    # --- Univariate ---
    # Numerical: describe() + skewness
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_stats = df[numeric_cols].describe().to_dict()
    skewness = df[numeric_cols].skew().to_dict()
    
    # Categorical: uniques + top frequencies
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    cat_stats = {col: {"uniques": df[col].nunique(), "top": df[col].mode()[0]} for col in cat_cols}
    
    # Dates: Range detection
    date_cols = df.select_dtypes(include=['datetime', 'object']).columns.tolist() # simplistic check
    date_ranges = {}
    for col in date_cols:
        try:
            temp_date = pd.to_datetime(df[col])
            date_ranges[col] = {"min": temp_date.min().strftime('%Y-%m-%d'), "max": temp_date.max().strftime('%Y-%m-%d')}
        except: continue

    # --- Bivariate: Target vs Rest ---
    correlations = {}
    if target in df.columns:
        if target in numeric_cols:
            # Pearson for numeric target
            correlations = df[numeric_cols].corr()[target].sort_values(ascending=False).to_dict()
        else:
            # For categorical targets, we look at group-by means or counts (simplified)
            correlations = "Categorical target: Analysis focuses on class distributions per feature."

    # --- Missing & Outliers ---
    missing = df.isnull().sum()[df.isnull().sum() > 0].to_dict()
    outlier_info = {}
    for col in numeric_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        outlier_count = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
        if outlier_count > 0: outlier_info[col] = int(outlier_count)

    # 3. LLM INTERPRETATION (The "Explanatory" Report)
    prompt = f"""
    Act as a Senior Data Auditor. Provide a MATHEMATICAL audit of this data for a **{task}** task.
    Target Variable: **{target}**
    
    RAW EVIDENCE:
    - Date Ranges: {date_ranges}
    - Numeric Stats (Mean/Std/Skew): {skewness}
    - Categorical Uniques: {cat_stats}
    - Bivariate (Target Correlation): {correlations}
    - Quality (Missing/Outliers): Missing: {missing}, Outliers: {outlier_info}
    
    YOUR MISSION:
    1. **Univariate Health**: Explain date coverage and flagging extreme skewness.
    2. **Bivariate Insight**: Which variables are strongest predictors or look like 'Data Leakage'?
    3. **Quality Alert**: State the impact of missing values/outliers on a {task} model.
    
    Output 4-5 bullet points. Be crisp, professional, and mathematically explanatory.
    """
    
    # llm = ChatOpenAI(openai_api_base="https://api.groq.com/openai/v1", model_name="llama-3.3-70b-versatile", temperature=0)
    response = llm.invoke(prompt)

    return {
        "quality_report": response.content,
        "messages": ["Auditor provided task-specific mathematical diagnostics."]
    }

In [23]:
test_state = {
    "file_path": "data/data.csv",
    "task_type": "Regression",
    "target_col": "Sales_USD"
}

print("🛡️ Running Quality Audit...")
result = auditor_node(test_state)
print("\n" + result["quality_report"])

🛡️ Running Quality Audit...


/var/folders/8l/qk1v82yj58z76rqcmg6l5v240000gn/T/ipykernel_1919/4256394068.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col])
/var/folders/8l/qk1v82yj58z76rqcmg6l5v240000gn/T/ipykernel_1919/4256394068.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col])
/var/folders/8l/qk1v82yj58z76rqcmg6l5v240000gn/T/ipykernel_1919/4256394068.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col])



As a Senior Data Auditor, I have conducted a mathematical audit of the provided data for a Regression task. Here are my findings:

* **Univariate Health**: The date range coverage is approximately 8 months, from '2024-01-01' to '2024-09-06', which may not be sufficient to capture seasonal or annual trends. The skewness of the target variable 'Sales_USD' is -0.0929, indicating a slightly left-skewed distribution. However, the skewness of 'Quantity' is -0.0405, which is relatively close to zero, suggesting a nearly symmetric distribution. No extreme skewness is observed.
* **Bivariate Insight**: The correlation between 'Sales_USD' and 'Quantity' is -0.00699, which is very weak. This suggests that 'Quantity' may not be a strong predictor of 'Sales_USD'. There is no apparent 'Data Leakage' in the provided bivariate correlation, as the correlation between the target variable and other variables is not extremely high.
* **Quality Alert**: Fortunately, there are no missing values in the data